# Phase 3 — Blind-A submission (INFERENCE ONLY)

Loads PRE-TRAINED artifacts (dense doc_mat, ColBERT PLAID index, a trained K2 LGBM) and runs the D1
spine to produce a CodaBench `prediction.json`. NO training or fine-tuning happens here — train K2 in
`phase2_rerank.ipynb` and point `K2_MODEL_PATH` at the result (`k2_lgbm.txt`).

Stack = RRF fusion (7 base channels + LoRA-ColBERT, focused-query routed) -> pre-trained K2 (LGBM). If
that K2 was trained with the frozen cross-encoder feature, the CE score is computed over the SERVE
candidates only at inference (a quick pass; never over any train split — there is no training here).

ONE flag drives the run:
- `BLIND=False` (dev-confirm): runs on the dev FINAL turns (Blind-A-comparable) and prints official nDCG@20.
- `BLIND=True` (submit): runs on talkpl-ai/TalkPlayData-Challenge-Blind-A (80 final-turn targets),
  applies the responder, writes the zip (`prediction.json` at root).

## 1. Drive + HF + Gemini auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME']=f'{DRIVE}/hf_cache'; OUT=f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'],exist_ok=True); os.makedirs(OUT,exist_ok=True)
try:
    t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=os.environ['HUGGINGFACE_HUB_TOKEN']=t
    from huggingface_hub import login; login(t); print('HF ok')
except Exception as e: print('no HF_TOKEN secret:', e)
try:                                   # only needed when RESPONDER=='gemini'
    os.environ['GEMINI_API_KEY']=os.environ.get('GEMINI_API_KEY') or userdata.get('GEMINI_API_KEY'); print('Gemini key ok')
except Exception as e: print('no GEMINI_API_KEY secret (only needed for the responder):', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas pylate peft google-genai
!pip -q uninstall -y torchao   # transformers wants torchao>0.16 but Colab ships 0.10 and its check RAISES; we don't use it
import sys; sys.path.insert(0,'.')

In [ ]:
# OOM hygiene — MUST run before any import that pulls in JAX/torch (RESTART to apply mid-session).
# Root cause of the GPU OOMs: JAX preallocates 75% of VRAM on init, leaving PyTorch ~24% -> OOM.
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'   # JAX: allocate on demand (THE fix)
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TORCHDYNAMO_DISABLE'] = '1'
print('OOM-hygiene env set — RESTART if torch/jax were already imported')

## 3. Config — set `BLIND` here

In [ ]:
# ── submission mode ──
BLIND=True                 # False = dev-confirm + print nDCG; True = Blind-A submit (spends a slot)
RESPONDER='gemini'         # 'gemini' (full composite, scripts/gemini_responder.py) | 'stub' (DEFAULT_RESPONSE, nDCG only)
DEFAULT_RESPONSE='ok'      # stub responder value (nDCG-only online eval; nb82 convention)
# Responder = scripts/gemini_responder.py (google-genai SDK). Best-of-N now runs a deterministic anti-pattern
# GUARD (fewest banned openers/crutch/reception/album-art wins, judge as tie-break) — fix for the 2026-06-21
# judge analysis (35% "Since" openers, 35% "you're looking for", hallucinated reception). Thinking is BUDGETED
# (not 0) so the model follows the variety/anti-crutch rules; the budget stays well under max_output (no empty-Part).
RESPONDER_MODEL='gemini-2.5-flash'        # generator (GEMINI_RESPONDER_MODEL). flash beat pro on the judge A/B (3.95 vs 3.75)
RESPONDER_TOP_N=1                         # tracks shown to the responder (nb80: 1 known-good; 3 dilutes)
RESPONDER_BEST_OF=2                       # generate N, GUARD picks the cleanest (judge tie-break). 3 gives the guard more to choose from
RESPONDER_JUDGE_MODEL='gemini-2.5-flash'  # judge for best-of-N tie-break — fast flash (thinking off); guard does the heavy lifting
RESPONDER_THINKING_BUDGET=512             # responder reasoning room so it obeys the opener-variety/anti-crutch rules (0 made flash lazy/templated)
RESPONDER_MAX_OUTPUT_TOKENS=2048          # must exceed the thinking budget; leaves ample room for the short reply
RESPONDER_CACHE=f'{OUT}/responder_cache'  # disk cache (rerun-free; keyed by prompt+model+best_of)

# ── two-step rerank (nDCG): stage 1 = K2 LGBM (coarse), stage 2 = LLM listwise reorder of the top-K ──
# Targets the addressable nDCG bucket (golds at K2-rank 21-50): the LLM reorders the K2 top-K so one can be
# promoted into the top-20. Default ON. Worth confirming on dev (BLIND=False, has gold) that it lifts ndcg@20
# before spending a Blind-A slot; set LISTWISE_RERANK=False for the K2-only (single-step) baseline.
LISTWISE_RERANK=True                      # K2 -> Gemini listwise reorder of top-K -> top-20 (default ON); False = K2 only
LISTWISE_WINDOW=50                        # how many of K2's top candidates the LLM reorders (must straddle the 20-cut)
LISTWISE_MODEL='gemini-2.5-flash'         # listwise reranker model
LISTWISE_THINKING_BUDGET=512              # ordering is a reasoning task: thinking ON but budgeted (avoids the empty-Part trap)
LISTWISE_CACHE=f'{OUT}/listwise_cache'    # disk cache (rerun-free; keyed by prompt)

SMOKE=0                    # >0 caps serve turns for a quick pipeline check (0 = all)
SEED=42

# ── splits (serve only — no training data is loaded in this notebook) ──
ORG='talkpl-ai'
DEV_SESSIONS=1000          # dev sessions for the dev-confirm serve set (final turn each)
BLIND_DATASET=f'{ORG}/TalkPlayData-Challenge-Blind-A'   # 80 sessions, NO gold

# ── retrieval ──
TOPK=500                   # fused-pool depth into the reranker
SUBMIT_K=20                # ids per submission row (official cap)
DENSE_MODEL='BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES={'cknn_audio':'audio-laion_clap','cknn_attr':'attributes-qwen3_embedding_0.6b'}
ENRICHED_GLOB=f'{OUT}/catalog_enriched_*.parquet'

# ── ColBERT (LoRA fine-tune; PLAID index built/loaded as an ARTIFACT, not trained here) ──
# D_LEN / EXPANSION_FIRST MUST match phase2_colbert_finetune (the checkpoint was trained + indexed at
# these) — serving a different doc length re-truncates docs vs how the model learned (train/serve skew).
COLBERT_OUT_DIR=f'{OUT}/colbert/music-colbert-v1'
Q_LEN=96; D_LEN=512; BSIZE=128; EXPANSION_FIRST=True
FT_IDX_FOLDER=f'{OUT}/colbert_plaid_ft'; FT_IDX_BASE='music-colbert-v1-d%d'%D_LEN

# ── PRE-TRAINED K2 (LGBM) — point at a trained artifact (.txt + .features.json sidecar) ──
# Train K2 in phase2_rerank.ipynb; THIS notebook only loads it (it writes k2_lgbm.txt by default). If the
# model used the frozen-CE feature, the serve-side CE is computed automatically over serve candidates only.
# (The new K2 feature set — interaction/consensus, per-turn <score>_norm, popularity_percentile, recency —
# is produced automatically by the shared FeatureBuilder below; the rerank() guard enforces train==serve.)
K2_MODEL_PATH=f'{OUT}/k2_lgbm.txt'   # the artifact phase2_rerank.ipynb saves; re-point if you keep a renamed K2
CE_MODEL='BAAI/bge-reranker-v2-m3'; CROSS_ENCODER_K=50; CE_MAX_DOC_TOKENS=480   # frozen-CE FEATURE depth (NOT K3b's 200)

import random as _r, numpy as _np
_r.seed(SEED); _np.random.seed(SEED)
try:
    import torch as _t; _t.manual_seed(SEED); _t.cuda.manual_seed_all(SEED)
except Exception: pass
assert os.path.exists(K2_MODEL_PATH), f'K2_MODEL_PATH missing: {K2_MODEL_PATH} — train K2 in phase2_rerank.ipynb first'
if BLIND and RESPONDER=='gemini':
    assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY missing — set the Colab secret before a blind+gemini run'
if LISTWISE_RERANK:
    assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY missing — listwise rerank (stage 2) needs it'
print('MODE:', 'BLIND-A SUBMIT' if BLIND else 'DEV-CONFIRM', '| responder', RESPONDER, '| best_of', RESPONDER_BEST_OF,
      '| listwise', LISTWISE_RERANK and f'w{LISTWISE_WINDOW}', '| D_LEN', D_LEN, '| K2', os.path.basename(K2_MODEL_PATH))

## 4. Catalog + base channels + dense doc_mat

In [ ]:
import glob, os, pickle, hashlib, pandas as pd, numpy as np
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion

meta_rows=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata',split='all_tracks')
_enr=sorted(glob.glob(ENRICHED_GLOB)); assert _enr, f'No enriched parquet at {ENRICHED_GLOB} — run A1 first'
_edf=pd.read_parquet(_enr[-1]); enr=dict(zip(_edf['track_id'],_edf['enriched_doc']))
cat=Catalog(meta_rows,enriched_docs=enr); USE_ENRICHED=True
print(f'enriched docs {len(enr)} (from {_enr[-1].split("/")[-1]})')

tre=load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings',split='all_tracks')
_avail=set(tre.column_names); CKNN_MODS={l:m for l,m in CONTENT_MODALITIES.items() if m in _avail}
te={l:TrackEmbeddings(tre.select_columns(['track_id',m]),modalities=[m]) for l,m in CKNN_MODS.items()}
te_cf=TrackEmbeddings(tre.select_columns(['track_id','cf-bpr']),modalities=['cf-bpr'])
ued=load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings'); ue=UserEmbeddings([r for sp in ued for r in ued[sp]])
dsd=load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')

model=SentenceTransformer(DENSE_MODEL,device='cuda')
# config-hashed cache key (model + enriched version + catalog size) — never silently load a stale matrix
_dm_sig=hashlib.md5(f'{DENSE_MODEL}|enriched={USE_ENRICHED}|{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}'.encode()).hexdigest()[:8]
DOC_MAT_NPY=f'{OUT}/dense_doc_mat_{_dm_sig}.npy'
if os.path.exists(DOC_MAT_NPY):
    doc_mat=np.load(DOC_MAT_NPY); print('loaded cached doc_mat', doc_mat.shape)
else:
    doc_mat=model.encode([cat.id_to_metadata(t,enriched=USE_ENRICHED) for t in cat.index_to_id],batch_size=256,normalize_embeddings=True,show_progress_bar=True)
    np.save(DOC_MAT_NPY, doc_mat); print('cached doc_mat ->', DOC_MAT_NPY)
dense=DenseChannel(cat.index_to_id,doc_mat,lambda qs:model.encode([DENSE_QUERY_PREFIX+q for q in qs],batch_size=256,normalize_embeddings=True),normalize=False)
COOC_PKL=f'{OUT}/artist_cooc.pkl'
cooc=pickle.load(open(COOC_PKL,'rb')) if os.path.exists(COOC_PKL) else build_artist_cooc(dsd['train'],tid_to_artists_from_catalog(cat))
if not os.path.exists(COOC_PKL): pickle.dump(cooc,open(COOC_PKL,'wb'))
cknn=[ContentKNNChannel(te[l],m,label=l) for l,m in CKNN_MODS.items()]
base_chans=[BM25Channel(cat,enriched=USE_ENRICHED), dense, *cknn,
            CFChannel(ue,te_cf,'cf-bpr'), SameArtistChannel(cat), RelatedArtistChannel(cat,cooc)]
qb_full=QueryBuilder()                    # base channels' serve query (full history)
qb_focused=QueryBuilder(recency_window=1) # ColBERT's focused query (train==serve)
print('base channels:', [c.label for c in base_chans])

## 5. ColBERT PLAID channel (focused-query routed)

In [ ]:
from pylate import models
from mcrs.training.colbert_index import build_or_load_plaid, colbert_retrieve
from mcrs.retrieval.colbert_channel import colbert_doc_text
import hashlib

assert os.path.isdir(COLBERT_OUT_DIR), f'ColBERT checkpoint missing at {COLBERT_OUT_DIR} — run phase2_colbert_finetune first'
ft=models.ColBERT(model_name_or_path=COLBERT_OUT_DIR, query_length=Q_LEN, document_length=D_LEN)
doc_fn=lambda t: colbert_doc_text(cat, t, expansion_first=EXPANSION_FIRST)
# checkpoint-keyed PLAID index name so a retrain can't reuse a stale index (mismatched embedding spaces)
_ck_sig=hashlib.md5('|'.join(f'{f}:{os.stat(os.path.join(rt,f)).st_size}:{int(os.stat(os.path.join(rt,f)).st_mtime)}'
                             for rt,_,fs in os.walk(COLBERT_OUT_DIR) for f in sorted(fs)).encode()).hexdigest()[:8]
# doc-side sig: enriched catalog + D_LEN + EXPANSION_FIRST determine the doc vectors. Identical format to
# phase2_colbert_finetune's gate so this REUSES the index the fine-tune built (no re-encode); a new enriched
# parquet / D_LEN / expansion order busts it. (D_LEN added so the 300->512 bump can't reuse a stale index.)
_doc_sig=hashlib.md5(f'{os.path.basename(_enr[-1])}|n={len(cat.index_to_id)}|d{D_LEN}|ef{EXPANSION_FIRST}'.encode()).hexdigest()[:8]
retr=build_or_load_plaid(ft, cat, FT_IDX_FOLDER, f'{FT_IDX_BASE}-{_ck_sig}-{_doc_sig}', doc_fn, batch_size=BSIZE)

class PlaidColBERTChannel:
    """Fusion channel backed by the fine-tuned ColBERT PLAID index. query_key='colbert' so RRF fusion
    routes it the focused query (per_channel_queries) while base channels keep the full query."""
    label='colbert'; query_key='colbert'
    def __init__(self, model, retriever): self.model, self.retriever = model, retriever
    def batch_text_to_item_retrieval(self, queries, topk, batch_context=None, user_ids=None):
        if not queries: return []
        return colbert_retrieve(self.model, self.retriever, list(queries), topk, batch_size=BSIZE)

colbert=PlaidColBERTChannel(ft, retr)
chans=base_chans+[colbert]
fusion=RRFFusion(chans, k=60)
labels=[c.label for c in chans]
PCQ={'colbert': qb_focused}               # per-channel routing: ColBERT gets the focused query
print('fusion channels:', labels)

## 6. Serve turns (final-turn targets)

In [ ]:
# Serve set ONLY — no training data is loaded in this notebook.
if BLIND:
    blind_rows=load_dataset(BLIND_DATASET, split='test')
    conv_serve=Conversations(blind_rows); has_gold=False
    serve_turns=list(conv_serve.target_turns())            # one trailing-turn target per session
    assert len(serve_turns)==len(blind_rows)==80, f'expected 80 Blind-A targets, got {len(serve_turns)}'
    print('BLIND: serve = Blind-A', len(serve_turns), 'final-turn targets')
else:
    conv_serve=Conversations(dsd['test'].select(range(min(DEV_SESSIONS,len(dsd['test']))))); has_gold=True
    serve_turns=list(conv_serve.target_turns())            # dev FINAL turns (Blind-A-comparable), with gold
    print('DEV-CONFIRM: serve =', len(serve_turns), 'dev final-turn targets')
if SMOKE: serve_turns=serve_turns[:SMOKE]; print('SMOKE: capped serve to', len(serve_turns))
assert len({t.session_id for t in serve_turns})==len(serve_turns), 'serve turns not one-per-session'
print('serve turns', len(serve_turns))

## 7. Load the pre-trained K2 + build serve features (inference only)

Loads a trained K2 by path (`K2_MODEL_PATH`). If that model used the frozen cross-encoder feature, the
CE score is computed over the SERVE candidates only — a quick pass, never over any train split. There
is no fitting in this notebook.

In [ ]:
import json as _json
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker

# which features does the trained K2 expect? (decides whether we need the frozen-CE serve pass)
feat_names=_json.load(open(K2_MODEL_PATH+'.features.json'))
need_ce=('ce_score' in feat_names)
print('K2 features:', len(feat_names), '| needs ce_score:', need_ce)

# routed serve pools (what K2 reranks) — ColBERT focused, base channels full
serve_bc=[{'history_tids':t.history_tids,'user_id':t.user_id} for t in serve_turns]
serve_pools=fusion.fuse([qb_full.build(t).text for t in serve_turns], TOPK, batch_context=serve_bc,
                        user_ids=[t.user_id for t in serve_turns],
                        per_channel_queries={'colbert':[qb_focused.build(t).text for t in serve_turns]})

# dense query<->candidate cosine feature (precompute serve query vecs)
_qcache={}
_qvm=model.encode([DENSE_QUERY_PREFIX+qb_full.build(t).text for t in serve_turns],batch_size=256,normalize_embeddings=True)
_qcache.update({(t.session_id,t.turn_number): _qvm[i] for i,t in enumerate(serve_turns)})
def dense_cos(ctx, tid):
    j=cat.id_to_index.get(tid)
    if j is None: return 0.0
    v=_qcache.get((ctx.session_id,ctx.turn_number))
    return float(v @ doc_mat[j]) if v is not None else 0.0
# full-conversation intent vector vs candidate doc (mirror phase2 chat_doc_cos)
_chatcache={}
_cvm=model.encode([DENSE_QUERY_PREFIX+((' '.join(list(t.utterances)+([t.goal] if t.goal else []))) or ' ') for t in serve_turns],batch_size=256,normalize_embeddings=True)
_chatcache.update({(t.session_id,t.turn_number): _cvm[i] for i,t in enumerate(serve_turns)})
def chat_doc_cos(ctx, tid):
    j=cat.id_to_index.get(tid)
    if j is None: return 0.0
    v=_chatcache.get((ctx.session_id,ctx.turn_number))
    return float(v @ doc_mat[j]) if v is not None else 0.0

# session vibe: mean doc-embedding of in-session played tracks vs candidate doc (mirror phase2 session_emb_cos)
def _sess_vec(ctx):
    idx=[cat.id_to_index[h] for h in ctx.history_tids if h in cat.id_to_index]
    if not idx: return None
    m=doc_mat[idx].mean(0); n=np.linalg.norm(m)
    return m/n if n>0 else None
def session_emb_cos(ctx, tid):
    j=cat.id_to_index.get(tid)
    if j is None: return 0.0
    sv=_sess_vec(ctx)
    return float(sv @ doc_mat[j]) if sv is not None else 0.0
# session vibe in audio-CLAP + CF-bpr space (idea 5: extend the proven doc-space session_emb_cos)
from mcrs.rerank.embedding_features import make_session_cos_fn, l2_normalize_rows
assert 'cknn_audio' in te, 'idea-5 needs the audio-laion_clap embedding (cknn_audio channel)'
_audio_te=te['cknn_audio']
session_audio_cos=make_session_cos_fn(_audio_te.id_to_index, l2_normalize_rows(_audio_te.matrix('audio-laion_clap')))
session_cf_cos=make_session_cos_fn(te_cf.id_to_index, l2_normalize_rows(te_cf.matrix('cf-bpr')))
score_fns={'dense_cos': dense_cos, 'chat_doc_cos': chat_doc_cos, 'session_emb_cos': session_emb_cos, 'session_audio_cos': session_audio_cos, 'session_cf_cos': session_cf_cos}

# frozen-CE feature: scored over the SERVE candidates ONLY (inference-time; this is NOT training)
if need_ce:
    from mcrs.rerank.neural import NeuralReranker, build_ce_score_lookup, make_ce_feature_fn
    from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn
    _fce=build_cross_encoder_score_fn(CE_MODEL,device='cuda',max_length=512,max_doc_tokens=CE_MAX_DOC_TOKENS,dtype='fp16')
    _nr=NeuralReranker(cat,qb_full,_fce,cross_encoder_k=CROSS_ENCODER_K,enriched=USE_ENRICHED)
    print('scoring frozen CE over serve pools (serve only)...')
    ce_lookup=build_ce_score_lookup(serve_turns, serve_pools, _nr, normalize=True, show_progress=True)
    score_fns['ce_score']=make_ce_feature_fn(ce_lookup, default=-1.0)

# LOAD the pre-trained K2 (no fit anywhere in this notebook)
fb=FeatureBuilder(cat, labels, score_fns=score_fns)
k2=LGBMReranker(fb).load(K2_MODEL_PATH)
print('loaded pre-trained K2 <-', K2_MODEL_PATH, '| features:', len(fb.feature_names))

## 8. Inference (rerank the serve pools) -> rows (+ nDCG when dev-confirm)

In [ ]:
from mcrs.contracts import SubmissionRow
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import validate_submission

_asm=TopKAssembler(cat, top_k=SUBMIT_K)

# Stage 2 (optional): LLM listwise reorder of K2's top-LISTWISE_WINDOW before the top-20 cut. One Gemini
# call per turn reorders the window (RankGPT-style); the LLM sees the conversation AND the in-session liked
# tracks (listwise_llm.render_conversation) and CLEANED candidate tags; any empty/unparseable output falls
# back to the K2 order (never drops a candidate). Disk-cached so reruns are free. NOTE: 1 call/turn.
if LISTWISE_RERANK:
    import sys, collections
    from mcrs.rerank.listwise_llm import listwise_rerank, make_gemini_generate_fn, cache_generate_fn
    sys.path.insert(0, 'scripts'); from gemini_responder import clean_tags as _clean_tags
    from tqdm.auto import tqdm
    _lw_gen=cache_generate_fn(
        make_gemini_generate_fn(os.environ['GEMINI_API_KEY'], model=LISTWISE_MODEL,
                                thinking_budget=LISTWISE_THINKING_BUDGET),
        LISTWISE_CACHE)
    def _f(v): return (v[0] if isinstance(v,list) and v else ('' if isinstance(v,list) else v)) or ''
    # tag doc-frequency + artist set (built once) so candidate descriptions use CLEANED tags — drops
    # ratings/playlist/opinion folksonomy junk that degraded the LLM ordering (bug #6); same cleaner the responder uses.
    _tagdf=collections.Counter(); _artists=set()
    for _tid in cat.index_to_id:
        _m=cat.metadata(_tid)
        for _t in {str(x).strip().lower() for x in (_m.get('tag_list') or [])}:
            if _t: _tagdf[_t]+=1
        _a=_f(_m.get('artist_name'))
        if _a: _artists.add(str(_a).strip().lower())
    def _describe(tid):                                   # compact 'Name by Artist (album) [clean tags]' for the LLM
        m=cat.metadata(tid)
        name=_f(m.get('track_name')); art=_f(m.get('artist_name')); alb=_f(m.get('album_name'))
        tags=_clean_tags(m.get('tag_list'), _tagdf, _artists, name, art)
        s=f"{name} by {art}"
        if alb: s+=f" (album {alb})"
        if tags: s+=f" [{', '.join(str(x) for x in tags)}]"
        return s[:240]

def _rank(t, pool):
    ranked=k2.rerank(t, pool)                             # stage 1: K2 LGBM coarse order over the fused pool
    if LISTWISE_RERANK:
        ranked=listwise_rerank(ranked, _describe, _lw_gen, window=LISTWISE_WINDOW)  # stage 2: LLM listwise
    return _asm.apply(ranked)

_iter=list(zip(serve_turns, serve_pools))
if LISTWISE_RERANK: _iter=tqdm(_iter, desc='listwise rerank')
rows=[SubmissionRow(t.session_id,t.user_id,t.turn_number,_rank(t,pool),'') for t,pool in _iter]
validate_submission(rows, catalog=cat, expected_keys=[(t.session_id,t.turn_number) for t in serve_turns])
print('rows:', len(rows), '| listwise', LISTWISE_RERANK)

if has_gold:
    from mcrs.eval.harness import GoldRow
    from mcrs.eval.official import score_official
    golds=[GoldRow(t.session_id,t.user_id,t.turn_number,conv_serve.gold(t.session_id,t.turn_number))
           for t in serve_turns if conv_serve.gold(t.session_id,t.turn_number) is not None]
    keyset={(g.session_id,g.turn_number) for g in golds}
    def _score(rws):
        rg=[r for r in rws if (r.session_id,r.turn_number) in keyset]
        return score_official(rg, golds, catalog_size=len(cat.index_to_id))
    sc=_score(rows)
    print('DEV-CONFIRM official:', {k:round(v,4) for k,v in sc.items() if 'ndcg' in k or 'diversity' in k})
    if LISTWISE_RERANK:
        # GATE (bug #7): measure the K2-only baseline so the listwise lift is VERIFIED, not assumed.
        rows_k2=[SubmissionRow(t.session_id,t.user_id,t.turn_number,_asm.apply(k2.rerank(t,pool)),'')
                 for t,pool in zip(serve_turns, serve_pools)]
        n_lw=sc.get('ndcg@20',float('nan')); n_k2=_score(rows_k2).get('ndcg@20',float('nan'))
        print(f'>>> GATE ndcg@20: K2-only {n_k2:.4f} -> +listwise {n_lw:.4f} (delta {n_lw-n_k2:+.4f})')
        print('>>> ship listwise to Blind ONLY if delta >= 0; else set LISTWISE_RERANK=False.')
    else:
        print('>>> K2-only baseline. Flip LISTWISE_RERANK=True to measure the stage-2 lift.')

## 9. Responder + package `prediction.json`

In [ ]:
import os, json, zipfile, datetime, dataclasses
from mcrs.run.harness import write_submission

_tag='blind' if BLIND else 'dev'
BASE=f'{OUT}/prediction_{_tag}_base.json'
# Base prediction (predicted_track_ids fixed). Stub path stamps DEFAULT_RESPONSE; gemini path lets the
# responder script rewrite predicted_response in place (track_ids untouched).
base_rows=[dataclasses.replace(r, predicted_response=DEFAULT_RESPONSE) for r in rows] if RESPONDER!='gemini' else list(rows)
write_submission(base_rows, BASE)
print('wrote base prediction ->', BASE, '| rows', len(base_rows))

FINAL=BASE
if RESPONDER=='gemini':
    # scripts/gemini_responder.py — google-genai SDK with THINKING DISABLED (--thinking-budget 0) + an
    # output-token cap: kills the gemini-2.5 empty-Part/finish_reason=STOP error (all budget spent thinking
    # -> .text raises -> fallback) and runs much faster. Run via `!` shell magic (like nb80) so the per-row
    # progress (N/80) STREAMS live in Colab (subprocess.run would buffer it until the end).
    os.environ['GEMINI_RESPONDER_MODEL']=RESPONDER_MODEL        # hard-set generator tier (script default is flash)
    os.environ['GEMINI_JUDGE_MODEL']=RESPONDER_JUDGE_MODEL
    DSET=BLIND_DATASET if BLIND else f'{ORG}/TalkPlayData-Challenge-Dataset'
    FINAL=f'{OUT}/prediction_{_tag}_gemini.json'
    !python -u scripts/gemini_responder.py --pred {BASE} --out {FINAL} --dataset {DSET} --top-n {RESPONDER_TOP_N} --best-of {RESPONDER_BEST_OF} --judge-model {RESPONDER_JUDGE_MODEL} --thinking-budget {RESPONDER_THINKING_BUDGET} --max-output-tokens {RESPONDER_MAX_OUTPUT_TOKENS} --sleep 0.2 --cache-dir {RESPONDER_CACHE} --fail-on-fallback
    assert os.path.exists(FINAL), 'gemini responder did not write output — read the traceback'
else:
    print(f"RESPONDER='stub': predicted_response defaulted to {DEFAULT_RESPONSE!r} (nDCG axis only; LLM axis floors)")

preds=json.load(open(FINAL))
if BLIND:
    # A scalar count can't tell a valid layout from a wrong one — verify ONE row per session AND that the
    # keys are exactly the official Blind-A target turns (serve_turns), and that no row ships zero ids.
    # The non-empty-response check also catches script fallbacks (a fallback keeps the empty base response).
    want={(t.session_id,t.turn_number) for t in serve_turns}
    got={(p['session_id'],p['turn_number']) for p in preds}
    assert len(preds)==80, f'expected 80 Blind-A rows, got {len(preds)} — DO NOT submit'
    assert len({p['session_id'] for p in preds})==80, 'rows not one-per-session — DO NOT submit'
    assert got==want, 'prediction keys != official Blind-A target turns — DO NOT submit'
    assert all(p['predicted_track_ids'] for p in preds), 'a row has empty predicted_track_ids — DO NOT submit'
    assert all((p['predicted_response'] or '').strip() for p in preds), 'a row has empty predicted_response (responder fallback) — DO NOT submit'
    print('blind checks OK: 80 rows, one-per-session, keys == official targets, ids+responses non-empty')
ZIP=f'{DRIVE}/recsys2026_submissions/{datetime.date.today().isoformat()}-{_tag}-fusion-colbert-k2ce.zip'
os.makedirs(os.path.dirname(ZIP),exist_ok=True)
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(FINAL, arcname='prediction.json')   # MUST be 'prediction.json' at zip ROOT for CodaBench
print('submission zip ready:', ZIP)
print('UPLOAD to CodaBench' if BLIND else 'DEV zip (not for upload — flip BLIND=True to submit)')